<a href="https://colab.research.google.com/github/kumarsirish/agentic-rag-workshop/blob/main/Movie-RAG-LangGraph.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<a href="https://colab.research.google.com/github/kumarsirish/agentic-rag-workshop/blob/main/Movie-RAG-LangGraph.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Movie Recommendation with RAG using LangGraph

An **agentic RAG** movie assistant. Instead of fixed steps, we draw a **graph** and let the LLM decide what to do.

LangGraph models an agent as boxes (**nodes**) joined by arrows (**edges**). Each node is a plain Python function.

**How it works**

- **Retrieval (the tool):** movie plots are turned into vectors by `SentenceTransformer` and stored in an in-memory **ChromaDB** collection. We wrap the search as a **tool** the LLM can call.
- **The agent (the graph):** Gemini reads the question and chooses: call the search tool, or answer. After each search the graph goes back to Gemini, so it can search again if the first results were not enough.
- **Grounding:** the instruction tells Gemini to answer using **only** the movies the tool returned.

```
 START
   |
   v
 agent  (Gemini)  ---- needs a search? ---->  tools  (search_movies_tool)
   ^                                              |
   |______________ results come back _____________|
   |
   | no more searches needed
   v
  END   (the answer)
```

What makes it *agentic*: the **LLM chooses the next step**, not our code.

## 0. Setup

In [1]:
# Run once. (In Colab this takes a couple of minutes.)
%pip install -q langgraph langchain-google-genai sentence-transformers chromadb gradio

In [2]:
import os
import json
import urllib.parse
import urllib.request

import chromadb
from sentence_transformers import SentenceTransformer
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_google_genai import ChatGoogleGenerativeAI
from langgraph.graph import StateGraph, MessagesState, START
from langgraph.prebuilt import ToolNode, tools_condition
from langgraph.checkpoint.memory import MemorySaver
from IPython.display import display, HTML

# Settings
EMBEDDING_MODEL = "all-MiniLM-L6-v2"     # turns text into 384 numbers
LLM_MODEL = "gemini-2.5-flash"           # the agent's brain
TOP_K = 5                                # how many movies the tool returns

In [3]:
# Keys are read from the Colab secrets vault (the key icon on the left).
# Outside Colab, or if the secret is missing, we ask you to paste the Gemini key instead.
from getpass import getpass


def load_colab_secret(name):
    """Copy a secret from the Colab vault into the environment. Return True if it worked."""
    try:
        from google.colab import userdata
        os.environ[name] = userdata.get(name)
        return True
    except Exception:
        return False


load_colab_secret("GOOGLE_API_KEY")                # Gemini key: https://aistudio.google.com/apikey
load_colab_secret("HF_TOKEN")                      # optional: speeds up the embedding model download

if "GOOGLE_API_KEY" not in os.environ:
    os.environ["GOOGLE_API_KEY"] = getpass("Paste your Gemini API key: ")

print("Google API key found:", "GOOGLE_API_KEY" in os.environ)
print("HF_TOKEN found      :", "HF_TOKEN" in os.environ)

Google API key found: True
HF_TOKEN found      : True


## 1. Our knowledge base

A small list of 32 movies. Each movie is a Python dictionary.
(Ratings are approximate, and the one-line reviews were written for this demo. They are not real quotes.)

In [4]:
movies = [
    {"title": "Inception", "year": 2010, "genre": "Sci-Fi, Thriller", "rating": 8.8,
     "cast": "Leonardo DiCaprio, Joseph Gordon-Levitt, Elliot Page",
     "overview": "A thief who steals secrets by entering people's dreams is hired to plant an idea in a target's mind.",
     "review": "Mind-bending and beautifully shot; you will want to watch it twice."},
    {"title": "Interstellar", "year": 2014, "genre": "Sci-Fi, Drama", "rating": 8.7,
     "cast": "Matthew McConaughey, Anne Hathaway, Jessica Chastain",
     "overview": "Astronauts travel through a wormhole to find a new home for humanity as Earth's crops fail.",
     "review": "Epic and emotional, with a score that stays with you."},
    {"title": "The Matrix", "year": 1999, "genre": "Sci-Fi, Action", "rating": 8.7,
     "cast": "Keanu Reeves, Laurence Fishburne, Carrie-Anne Moss",
     "overview": "A hacker discovers that reality is a simulation and joins a rebellion against the machines.",
     "review": "Groundbreaking action and a story that makes you question reality."},
    {"title": "Toy Story", "year": 1995, "genre": "Animation, Family", "rating": 8.3,
     "cast": "Tom Hanks, Tim Allen, Don Rickles",
     "overview": "A cowboy doll feels threatened when a new space ranger toy arrives in a boy's room.",
     "review": "Warm, funny and full of heart for kids and adults alike."},
    {"title": "Finding Nemo", "year": 2003, "genre": "Animation, Family", "rating": 8.2,
     "cast": "Albert Brooks, Ellen DeGeneres, Alexander Gould",
     "overview": "A timid clownfish crosses the ocean to rescue his son.",
     "review": "A colorful, touching adventure about a father's love."},
    {"title": "The Shawshank Redemption", "year": 1994, "genre": "Drama", "rating": 9.3,
     "cast": "Tim Robbins, Morgan Freeman, Bob Gunton",
     "overview": "A banker wrongly jailed for murder finds hope and friendship over decades in prison.",
     "review": "A quiet, powerful story about hope that lifts you up."},
    {"title": "The Godfather", "year": 1972, "genre": "Crime, Drama", "rating": 9.2,
     "cast": "Marlon Brando, Al Pacino, James Caan",
     "overview": "The aging head of a crime family hands control of the empire to his reluctant son.",
     "review": "Slow-burning, masterful storytelling about family and power."},
    {"title": "The Dark Knight", "year": 2008, "genre": "Action, Crime", "rating": 9.0,
     "cast": "Christian Bale, Heath Ledger, Aaron Eckhart",
     "overview": "Batman faces the Joker, a chaotic criminal who pushes Gotham City to its limits.",
     "review": "Dark and intense, with an unforgettable villain."},
    {"title": "Forrest Gump", "year": 1994, "genre": "Drama, Romance", "rating": 8.8,
     "cast": "Tom Hanks, Robin Wright, Gary Sinise",
     "overview": "A kind man with a low IQ unknowingly takes part in major moments of American history.",
     "review": "Gentle, moving and quietly funny; a real feel-good classic."},
    {"title": "Parasite", "year": 2019, "genre": "Thriller, Drama", "rating": 8.5,
     "cast": "Song Kang-ho, Lee Sun-kyun, Cho Yeo-jeong",
     "overview": "A poor family schemes to get jobs in a wealthy household, with unexpected consequences.",
     "review": "Sharp, darkly funny and full of surprises."},
    {"title": "Spirited Away", "year": 2001, "genre": "Animation, Fantasy", "rating": 8.6,
     "cast": "Rumi Hiiragi, Miyu Irino, Mari Natsuki",
     "overview": "A girl wanders into a world of spirits and must work in a bathhouse to free her parents.",
     "review": "Imaginative and magical; every frame feels hand-crafted."},
    {"title": "The Lion King", "year": 1994, "genre": "Animation, Family", "rating": 8.5,
     "cast": "Matthew Broderick, Jeremy Irons, James Earl Jones",
     "overview": "A young lion prince flees after his father's death and later returns to reclaim his kingdom.",
     "review": "A timeless, emotional story with wonderful songs."},
    {"title": "Mad Max: Fury Road", "year": 2015, "genre": "Action, Sci-Fi", "rating": 8.1,
     "cast": "Tom Hardy, Charlize Theron, Nicholas Hoult",
     "overview": "In a desert wasteland, a drifter and a rebel driver flee a tyrant in one long chase.",
     "review": "Non-stop, breathtaking action with very little dialogue."},
    {"title": "La La Land", "year": 2016, "genre": "Romance, Musical", "rating": 8.0,
     "cast": "Ryan Gosling, Emma Stone, John Legend",
     "overview": "A jazz pianist and an aspiring actress fall in love while chasing their dreams in Los Angeles.",
     "review": "Dreamy and romantic with a bittersweet, memorable ending."},
    {"title": "The Grand Budapest Hotel", "year": 2014, "genre": "Comedy, Adventure", "rating": 8.1,
     "cast": "Ralph Fiennes, Tony Revolori, F. Murray Abraham",
     "overview": "A legendary hotel concierge and his young lobby boy are pulled into the theft of a priceless painting.",
     "review": "Quirky, stylish and laugh-out-loud funny."},
    {"title": "Whiplash", "year": 2014, "genre": "Drama, Music", "rating": 8.5,
     "cast": "Miles Teller, J.K. Simmons, Melissa Benoist",
     "overview": "A young drummer is pushed to his limits by a ruthless music teacher.",
     "review": "Tense and electrifying; you will be gripping your seat."},
    {"title": "Get Out", "year": 2017, "genre": "Horror, Thriller", "rating": 7.7,
     "cast": "Daniel Kaluuya, Allison Williams, Bradley Whitford",
     "overview": "A young man visiting his girlfriend's family slowly uncovers a disturbing secret.",
     "review": "Clever, creepy social thriller that keeps you guessing."},
    {"title": "Coco", "year": 2017, "genre": "Animation, Family", "rating": 8.4,
     "cast": "Anthony Gonzalez, Gael Garcia Bernal, Benjamin Bratt",
     "overview": "A boy who dreams of becoming a musician enters the Land of the Dead to uncover his family's history.",
     "review": "Colorful and full of music; keep tissues nearby."},
    {"title": "The Martian", "year": 2015, "genre": "Sci-Fi, Adventure", "rating": 8.0,
     "cast": "Matt Damon, Jessica Chastain, Chiwetel Ejiofor",
     "overview": "An astronaut stranded on Mars uses science and humor to survive until rescue.",
     "review": "Funny, optimistic and a joy for science lovers."},
    {"title": "Arrival", "year": 2016, "genre": "Sci-Fi, Drama", "rating": 7.9,
     "cast": "Amy Adams, Jeremy Renner, Forest Whitaker",
     "overview": "A linguist works to communicate with alien visitors before fear pushes the world toward war.",
     "review": "Thoughtful and moving; a slow-burn sci-fi that rewards patience."},
    {"title": "Back to the Future", "year": 1985, "genre": "Sci-Fi, Comedy", "rating": 8.5,
     "cast": "Michael J. Fox, Christopher Lloyd, Lea Thompson",
     "overview": "A teenager is sent thirty years into the past in a time-traveling car and must make sure his parents fall in love.",
     "review": "Pure fun: clever, fast and endlessly rewatchable."},
    {"title": "Amelie", "year": 2001, "genre": "Romance, Comedy", "rating": 8.3,
     "cast": "Audrey Tautou, Mathieu Kassovitz, Rufus",
     "overview": "A shy Parisian waitress secretly improves the lives of the people around her while searching for love.",
     "review": "Charming, whimsical and uplifting."},
    {"title": "Gladiator", "year": 2000, "genre": "Action, Drama", "rating": 8.5,
     "cast": "Russell Crowe, Joaquin Phoenix, Connie Nielsen",
     "overview": "A betrayed Roman general becomes a gladiator and seeks revenge on the corrupt emperor.",
     "review": "Grand, gritty and emotional with a powerful lead."},
    {"title": "The Silence of the Lambs", "year": 1991, "genre": "Thriller, Crime", "rating": 8.6,
     "cast": "Jodie Foster, Anthony Hopkins, Scott Glenn",
     "overview": "A young FBI trainee asks an imprisoned psychiatrist for help in catching a serial killer.",
     "review": "Chilling and suspenseful with unforgettable performances."},
    {"title": "Up", "year": 2009, "genre": "Animation, Adventure", "rating": 8.3,
     "cast": "Ed Asner, Jordan Nagai, Christopher Plummer",
     "overview": "An elderly widower ties balloons to his house and flies to South America with an unexpected young stowaway.",
     "review": "The opening alone will make you cry; the rest will make you smile."},
    {"title": "Knives Out", "year": 2019, "genre": "Mystery, Comedy", "rating": 7.9,
     "cast": "Daniel Craig, Chris Evans, Ana de Armas",
     "overview": "A detective investigates the death of a wealthy novelist surrounded by a family of suspects.",
     "review": "A witty, twisty whodunit that is great fun to guess along with."},
    {"title": "Everything Everywhere All at Once", "year": 2022, "genre": "Sci-Fi, Comedy", "rating": 7.8,
     "cast": "Michelle Yeoh, Stephanie Hsu, Ke Huy Quan",
     "overview": "A laundromat owner is pulled across parallel universes to stop a threat that could unravel reality.",
     "review": "Wild, inventive and surprisingly touching."},
    {"title": "The Pursuit of Happyness", "year": 2006, "genre": "Drama", "rating": 8.0,
     "cast": "Will Smith, Jaden Smith, Thandiwe Newton",
     "overview": "A struggling salesman and his young son face homelessness while he works toward a career in finance.",
     "review": "Inspiring and heartfelt, anchored by a warm father-son bond."},
    {"title": "Dune", "year": 2021, "genre": "Sci-Fi, Adventure", "rating": 8.0,
     "cast": "Timothee Chalamet, Rebecca Ferguson, Oscar Isaac",
     "overview": "A noble young heir travels to a dangerous desert planet to protect his family and its people.",
     "review": "Visually stunning and immersive on a big screen."},
    {"title": "Paddington 2", "year": 2017, "genre": "Family, Comedy", "rating": 7.8,
     "cast": "Ben Whishaw, Hugh Bonneville, Sally Hawkins",
     "overview": "A polite bear is wrongly jailed for a theft and his adoptive family works to prove his innocence.",
     "review": "Sweet, funny and kind-hearted; pure comfort viewing."},
    {"title": "Your Name", "year": 2016, "genre": "Animation, Romance", "rating": 8.4,
     "cast": "Ryunosuke Kamiki, Mone Kamishiraishi, Ryo Narita",
     "overview": "Two teenagers mysteriously swap bodies and try to find each other across distance and time.",
     "review": "Gorgeous animation and a romance that hits hard."},
    {"title": "Oppenheimer", "year": 2023, "genre": "Drama, History", "rating": 8.3,
     "cast": "Cillian Murphy, Emily Blunt, Matt Damon",
     "overview": "The story of the physicist who led the creation of the first atomic bomb and of its aftermath.",
     "review": "Dense, intense and gripping; a big, serious drama."},
]

print("Movies in our knowledge base:", len(movies))
movies[0]

Movies in our knowledge base: 32


{'title': 'Inception',
 'year': 2010,
 'genre': 'Sci-Fi, Thriller',
 'rating': 8.8,
 'cast': 'Leonardo DiCaprio, Joseph Gordon-Levitt, Elliot Page',
 'overview': "A thief who steals secrets by entering people's dreams is hired to plant an idea in a target's mind.",
 'review': 'Mind-bending and beautifully shot; you will want to watch it twice.'}

## 2. Turn each movie into text

The embedding model reads text, so we describe every movie as one string: title, year, genre, rating, cast, plot and review. Everything we want to be searchable goes in.
The same string is also what the agent reads when the search tool returns a movie.

In [5]:
def movie_to_text(movie):
    """Describe one movie as a single line of text."""
    return (f"{movie['title']} ({movie['year']}) | Genre: {movie['genre']} | Rating: {movie['rating']} "
            f"| Cast: {movie['cast']} | Plot: {movie['overview']} | Review: {movie['review']}")


movie_texts = []
for movie in movies:
    movie_texts.append(movie_to_text(movie))

print(movie_texts[3])

Toy Story (1995) | Genre: Animation, Family | Rating: 8.3 | Cast: Tom Hanks, Tim Allen, Don Rickles | Plot: A cowboy doll feels threatened when a new space ranger toy arrives in a boy's room. | Review: Warm, funny and full of heart for kids and adults alike.


## 3. Embeddings and the ChromaDB vector store

An **embedding** is a list of numbers that captures the *meaning* of a text. Texts with similar meaning get similar numbers, so "a funny cartoon for kids" ends up close to *Toy Story*, even though they share no words.

**ChromaDB** is a vector database. It stores the vectors together with each movie's details and quickly finds the closest ones.
The first run downloads the embedding model (about 90 MB).

In [6]:
embedder = SentenceTransformer(EMBEDDING_MODEL)
movie_vectors = embedder.encode(movie_texts)

print("Shape:", movie_vectors.shape)                       # (32 movies, 384 numbers each)
print("First 5 numbers of movie 0:", movie_vectors[0][:5])

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Shape: (32, 384)
First 5 numbers of movie 0: [-0.12738837 -0.04504668  0.02296315 -0.01150418  0.03554049]


In [7]:
def build_collection(movies, vectors, texts):
    """Create an in-memory ChromaDB collection holding our movies and their vectors."""
    client = chromadb.Client()
    collection = client.get_or_create_collection(name="movie_collection")

    collection.add(
        ids=[str(number) for number in range(len(movies))],
        embeddings=vectors.tolist(),        # we computed the vectors ourselves
        documents=texts,
        metadatas=movies,                   # each movie dictionary is stored next to its vector
    )
    return collection


collection = build_collection(movies, movie_vectors, movie_texts)
print("Movies stored in ChromaDB:", collection.count())

Movies stored in ChromaDB: 32


## 4. Retrieval function

Embed the question, then ask ChromaDB for the closest movies. A **smaller distance means a closer match**.

In [8]:
def search_movies_chroma(question, k=TOP_K):
    """Return the k movies whose meaning is closest to the question."""
    question_vector = embedder.encode([question]).tolist()
    results = collection.query(query_embeddings=question_vector, n_results=k)

    found = []
    for movie, distance in zip(results["metadatas"][0], results["distances"][0]):
        movie = dict(movie)
        movie["distance"] = round(float(distance), 3)
        found.append(movie)
    return found


def show_movies(found):
    """Print the movies as simple info cards."""
    for movie in found:
        print(f"{movie['title']} ({movie['year']})  |  Rating: {movie['rating']}  |  {movie['genre']}  |  distance {movie['distance']}")
        print(f"   Cast:    {movie['cast']}")
        print(f"   Summary: {movie['overview']}")
        print(f"   Review:  {movie['review']}")
        print()


show_movies(search_movies_chroma("a funny animated movie the whole family can enjoy", 3))

Up (2009)  |  Rating: 8.3  |  Animation, Adventure  |  distance 1.248
   Cast:    Ed Asner, Jordan Nagai, Christopher Plummer
   Summary: An elderly widower ties balloons to his house and flies to South America with an unexpected young stowaway.
   Review:  The opening alone will make you cry; the rest will make you smile.

Paddington 2 (2017)  |  Rating: 7.8  |  Family, Comedy  |  distance 1.323
   Cast:    Ben Whishaw, Hugh Bonneville, Sally Hawkins
   Summary: A polite bear is wrongly jailed for a theft and his adoptive family works to prove his innocence.
   Review:  Sweet, funny and kind-hearted; pure comfort viewing.

Everything Everywhere All at Once (2022)  |  Rating: 7.8  |  Sci-Fi, Comedy  |  distance 1.355
   Cast:    Michelle Yeoh, Stephanie Hsu, Ke Huy Quan
   Summary: A laundromat owner is pulled across parallel universes to stop a threat that could unravel reality.
   Review:  Wild, inventive and surprisingly touching.



## 5. Turn retrieval into a **tool**

A tool is a normal Python function that the LLM is allowed to call. The `@tool` decorator tells LangChain to read three things from it:

1. the **function name**
2. the **docstring**, which tells the LLM *when* to use it
3. the **type hints**, which tell the LLM what to pass in

So the docstring matters: it is the instruction manual for the tool.

The tool also remembers the movies it found in `last_found`, so we can show posters later.

In [9]:
last_found = []      # the movies from the most recent search


@tool
def search_movies_tool(question: str) -> str:
    """Search the movie database. Use this to find movies by genre, cast, plot, mood or review.

    Args:
        question: What the user is looking for, for example "animated family movies".
    """
    global last_found
    last_found = search_movies_chroma(question)

    result = ""
    for movie in last_found:
        result = result + "- " + movie_to_text(movie) + "\n"
    return result


# A tool is just a function, so we can test it without any LLM.
print(search_movies_tool.invoke({"question": "space adventure"})[:400])

- The Martian (2015) | Genre: Sci-Fi, Adventure | Rating: 8.0 | Cast: Matt Damon, Jessica Chastain, Chiwetel Ejiofor | Plot: An astronaut stranded on Mars uses science and humor to survive until rescue. | Review: Funny, optimistic and a joy for science lovers.
- Up (2009) | Genre: Animation, Adventure | Rating: 8.3 | Cast: Ed Asner, Jordan Nagai, Christopher Plummer | Plot: An elderly widower ties


## 6. Build the graph

A LangGraph app has four ingredients:

| Part | What it is |
|---|---|
| **State** | the data passed between nodes. `MessagesState` is simply the list of chat messages. |
| **Nodes** | Python functions. Each reads the state and returns something to add to it. |
| **Edges** | arrows that say which node runs next. |
| **Conditional edge** | an arrow that *chooses* the next node. `tools_condition` looks at the LLM's reply: if it asked for a tool go to `tools`, otherwise finish. |

We need only **two nodes**:

- `agent`: sends the conversation to Gemini. The reply is either a tool request or a final answer.
- `tools`: runs the tool the LLM asked for (LangGraph's ready-made `ToolNode`).

The instruction below plays the same role as our hand-written prompt in simple RAG.

In [10]:
llm = ChatGoogleGenerativeAI(model=LLM_MODEL)
llm_with_tools = llm.bind_tools([search_movies_tool])      # tell the LLM which tools exist

INSTRUCTION = (
    "You are a friendly movie assistant. "
    "ALWAYS use the search_movies_tool to find movies before answering. "
    "Answer using ONLY the movies the tool returns. "
    "If the user wants recommendations, pick the best 3 and give one short reason for each. "
    "If the tool results do not contain the answer, say so honestly."
)


def agent_node(state):
    """Ask the LLM what to do next: call the tool, or answer."""
    messages = [SystemMessage(INSTRUCTION)] + state["messages"]
    reply = llm_with_tools.invoke(messages)
    return {"messages": [reply]}


graph = StateGraph(MessagesState)
graph.add_node("agent", agent_node)
graph.add_node("tools", ToolNode([search_movies_tool]))

graph.add_edge(START, "agent")                        # start by asking the agent
graph.add_conditional_edges("agent", tools_condition) # tool request -> "tools", otherwise -> end
graph.add_edge("tools", "agent")                      # after a search, go back to the agent

app = graph.compile(checkpointer=MemorySaver())       # MemorySaver = remember the conversation

print(app.get_graph().draw_mermaid())                 # paste this into https://mermaid.live to see the picture

---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	agent(agent)
	tools(tools<hr/><small><em>__tools = search_movies_tool</em></small>)
	__end__([<p>__end__</p>]):::last
	__start__ --> agent;
	agent -.-> __end__;
	agent -.-> tools;
	tools --> agent;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc



## 7. Run the graph

`app.invoke(...)` runs the graph from START until it reaches the end and returns the final state. The last message in it is the answer.

The `thread_id` names the conversation. Using the same id again makes the agent remember earlier questions, so try a follow-up like *"which of those is the shortest?"*.

In [11]:
def ask_agent(question):
    """Send a question through the graph and return the final answer as text."""
    config = {"configurable": {"thread_id": "student-1"}}
    result = app.invoke({"messages": [HumanMessage(question)]}, config)
    return result["messages"][-1].text

In [12]:
question = "Something like Interstellar, but shorter and more hopeful"

answer = ask_agent(question)

print("Question:", question)
print()
print("Movies the tool found:")
show_movies(last_found)
print("Agent's answer:")
print(answer)

Question: Something like Interstellar, but shorter and more hopeful

Movies the tool found:
Interstellar (2014)  |  Rating: 8.7  |  Sci-Fi, Drama  |  distance 1.012
   Cast:    Matthew McConaughey, Anne Hathaway, Jessica Chastain
   Summary: Astronauts travel through a wormhole to find a new home for humanity as Earth's crops fail.
   Review:  Epic and emotional, with a score that stays with you.

Arrival (2016)  |  Rating: 7.9  |  Sci-Fi, Drama  |  distance 1.263
   Cast:    Amy Adams, Jeremy Renner, Forest Whitaker
   Summary: A linguist works to communicate with alien visitors before fear pushes the world toward war.
   Review:  Thoughtful and moving; a slow-burn sci-fi that rewards patience.

Oppenheimer (2023)  |  Rating: 8.3  |  Drama, History  |  distance 1.302
   Cast:    Cillian Murphy, Emily Blunt, Matt Damon
   Summary: The story of the physicist who led the creation of the first atomic bomb and of its aftermath.
   Review:  Dense, intense and gripping; a big, serious drama.

**Your turn:** change the question and run it again. Try a genre, an actor, or a mood.

In [13]:
answer = ask_agent("Movies with Tom Hanks")

show_movies(last_found)
print(answer)

Forrest Gump (1994)  |  Rating: 8.8  |  Drama, Romance  |  distance 0.969
   Cast:    Tom Hanks, Robin Wright, Gary Sinise
   Summary: A kind man with a low IQ unknowingly takes part in major moments of American history.
   Review:  Gentle, moving and quietly funny; a real feel-good classic.

Toy Story (1995)  |  Rating: 8.3  |  Animation, Family  |  distance 1.095
   Cast:    Tom Hanks, Tim Allen, Don Rickles
   Summary: A cowboy doll feels threatened when a new space ranger toy arrives in a boy's room.
   Review:  Warm, funny and full of heart for kids and adults alike.

The Grand Budapest Hotel (2014)  |  Rating: 8.1  |  Comedy, Adventure  |  distance 1.227
   Cast:    Ralph Fiennes, Tony Revolori, F. Murray Abraham
   Summary: A legendary hotel concierge and his young lobby boy are pulled into the theft of a priceless painting.
   Review:  Quirky, stylish and laugh-out-loud funny.

Oppenheimer (2023)  |  Rating: 8.3  |  Drama, History  |  distance 1.251
   Cast:    Cillian Murphy, 

### Watch the graph run

`app.stream(...)` gives us one update per node. Printing them shows the path through the graph: agent, tools, agent. This is the part simple RAG did not have.

In [14]:
config = {"configurable": {"thread_id": "student-2"}}
query = "Recommed a romantic movie"
#query = "what is the news today?"
for step in app.stream({"messages": [HumanMessage(query)]}, config, stream_mode="updates"):
    for node_name, update in step.items():
        message = update["messages"][-1]
        if node_name == "tools":
            print("NODE tools : returned movies")
            show_movies(last_found)
        elif message.tool_calls:
            print("NODE agent : calls tool", message.tool_calls[0]["name"], message.tool_calls[0]["args"])
        else:
            print("NODE agent :", message.text[:300])

NODE agent : calls tool search_movies_tool {'question': 'romantic movies'}
NODE tools : returned movies
La La Land (2016)  |  Rating: 8.0  |  Romance, Musical  |  distance 1.205
   Cast:    Ryan Gosling, Emma Stone, John Legend
   Summary: A jazz pianist and an aspiring actress fall in love while chasing their dreams in Los Angeles.
   Review:  Dreamy and romantic with a bittersweet, memorable ending.

The Grand Budapest Hotel (2014)  |  Rating: 8.1  |  Comedy, Adventure  |  distance 1.297
   Cast:    Ralph Fiennes, Tony Revolori, F. Murray Abraham
   Summary: A legendary hotel concierge and his young lobby boy are pulled into the theft of a priceless painting.
   Review:  Quirky, stylish and laugh-out-loud funny.

Get Out (2017)  |  Rating: 7.7  |  Horror, Thriller  |  distance 1.421
   Cast:    Daniel Kaluuya, Allison Williams, Bradley Whitford
   Summary: A young man visiting his girlfriend's family slowly uncovers a disturbing secret.
   Review:  Clever, creepy social thriller that

## Show the posters

This part is only for looks, it is not RAG. It looks up each poster on OMDb, then on TMDb, and shows a plain picture if neither has it.
Both services give free API keys (https://www.omdbapi.com/apikey.aspx and https://www.themoviedb.org/settings/api). You can use your own.

In [15]:
OMDB_KEY = "fc1fef96"
TMDB_KEY = "f5a0494cf81e7d1e1f13b65a953af255"


def read_json(url):
    """Download a web address and return its JSON content."""
    with urllib.request.urlopen(url, timeout=5) as response:
        return json.loads(response.read().decode())


def poster_from_omdb(title, year):
    """Return a poster url from OMDb, or None."""
    url = f"https://www.omdbapi.com/?t={urllib.parse.quote(title)}&y={year}&apikey={OMDB_KEY}"
    try:
        data = read_json(url)
        if data.get("Poster", "N/A").startswith("http"):
            return data["Poster"]
    except Exception as error:
        print("OMDb failed for", title, "->", error)
    return None


def poster_from_tmdb(title, year):
    """Return a poster url from TMDb, or None."""
    url = f"https://api.themoviedb.org/3/search/movie?api_key={TMDB_KEY}&query={urllib.parse.quote(title)}&year={year}"
    try:
        results = read_json(url).get("results", [])
        if len(results) > 0 and results[0].get("poster_path"):
            return "https://image.tmdb.org/t/p/w300" + results[0]["poster_path"]
    except Exception as error:
        print("TMDb failed for", title, "->", error)
    return None


def get_poster_url(title, year):
    """Try OMDb, then TMDb, then a plain placeholder picture."""
    poster = poster_from_omdb(title, year)
    if poster is None:
        poster = poster_from_tmdb(title, year)
    if poster is None:
        poster = "https://placehold.co/180x260/0f172a/ffffff?text=" + urllib.parse.quote(title)
    return poster


def movies_to_html(found):
    """Build one HTML card per movie, with the poster on the left."""
    cards = ""
    for movie in found:
        poster = get_poster_url(movie["title"], movie["year"])
        cards = cards + f"""
        <div style="display: flex; margin-bottom: 16px; border: 1px solid #ddd; padding: 12px; border-radius: 8px; background: #f9f9f9; color: #222;">
            <img src="{poster}" alt="{movie['title']} poster" style="width: 180px; margin-right: 18px; border-radius: 6px; object-fit: cover;"
                 onerror="this.src='https://placehold.co/180x260?text=No+Image'">
            <div>
                <h3 style="margin: 0; font-size: 1.35em;">{movie['title']} ({movie['year']})</h3>
                <p style="margin: 6px 0;"><b>Rating:</b> ⭐ {movie['rating']} | <b>Genre:</b> {movie['genre']}</p>
                <p style="margin: 6px 0;"><b>Cast:</b> {movie['cast']}</p>
                <p style="margin: 6px 0;">{movie['overview']}</p>
                <p style="margin: 6px 0; font-style: italic; color: #555;">"{movie['review']}"</p>
            </div>
        </div>
        """
    return cards




## Tiny web app with Gradio

The same `ask_agent` function behind a text box.

In [16]:
import gradio as gr

def web_search(question):
    """What the web app runs when someone presses Submit."""
    answer = ask_agent(question)

    # Filter last_found to keep only the movies mentioned in the agent's final text response
    recommended_movies = []
    for movie in last_found:
        # Check if the title is mentioned in the agent's final answer (case-insensitive)
        if movie["title"].lower() in answer.lower():
            recommended_movies.append(movie)

    # If no titles matched for some reason, default back to last_found to avoid a blank display
    if not recommended_movies:
        recommended_movies = last_found

    return answer, movies_to_html(recommended_movies)

demo = gr.Interface(
    fn=web_search,
    inputs=gr.Textbox(label="What kind of movie do you want?", placeholder="e.g. a heartwarming animated movie for the whole family"),
    outputs=[gr.Markdown(label="Answer"), gr.HTML(label="Movies")],
    title="🎬 Movie RAG Agent (LangGraph)",
)

demo.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://500d4573c07c52235d.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


## Simple RAG vs LangGraph agent

| | Simple RAG | LangGraph agent |
|---|---|---|
| Who decides to search? | our code, always | the LLM, through a conditional edge |
| Who writes the prompt? | `build_prompt` | the `INSTRUCTION` text |
| Flow of steps | fixed line of function calls | a graph we draw: nodes and edges |
| Search query | the user's exact words | the LLM can rephrase it |
| Several searches per question? | no | yes, the graph loops back to the agent |
| Follow-up questions | needs extra code | `MemorySaver` + `thread_id` |
| Cost | 1 LLM call | 2 or more LLM calls |

## What to try next

- **Tool docstring:** make it vague (`"Search stuff."`) and see whether the LLM still calls it.
- **Ask something that needs no search** (for example *"hello"*). Does the graph skip the `tools` node?
- **Add a third node**, for example one that checks whether the answer used only retrieved movies, and draw an edge to it.
- **Add a second tool**, for example `get_top_rated(genre)`, and add it to both `bind_tools` and `ToolNode`.
- **Add a rewrite node:** if the search results look weak, let the LLM rewrite the question and search again, then draw the new edges.